# MNIST CNN

Digit classification on MNIST with a small CNN: two blocks of Conv2d, BatchNorm2d, ReLU, Dropout and MaxPool2d, then a Linear layer trained with CrossEntropyLoss and Adam. It is the architecture of the MNIST CNN example of synapgrad, the first project of the series. Set `engine_str` to `'raxis'` or `'torch'`. torchvision is only used to download the dataset.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# Make `examples.utils` importable from this directory
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
from torchvision import datasets

In [ ]:
from examples.utils.data import split_dataset, DataLoader, DataLoaderCallback
from examples.utils.train import Trainer, Evaluator

In [ ]:
supported_engines = ['raxis', 'torch']

In [ ]:
seed = 42
engine_str = 'raxis'
device = 'cpu'

In [ ]:
epochs = 5
batch_size = 128
lr = 0.01

In [ ]:
if engine_str == 'raxis':
    import raxis as engine
    from raxis import nn, optim
elif engine_str == 'torch':
    import torch as engine
    from torch import nn, optim
else:
    raise ValueError(f"engine_str must be one of {supported_engines}")

engine.manual_seed(seed)
np.random.seed(seed)
print("Engine:", engine_str)

In [ ]:
train_set = datasets.MNIST('./mnist', train=True, download=True)
test_set = datasets.MNIST('./mnist', train=False, download=True)

trainX = train_set.data.numpy()
trainY = train_set.targets.numpy()
testX = test_set.data.numpy()
testY = test_set.targets.numpy()

(trainX, trainY), (valX, valY), _ = split_dataset(trainX, trainY, test_split=0.2)

# (N, 28, 28) -> (N, 1, 28, 28): one input channel
trainX = np.expand_dims(trainX / 255.0, axis=1)
valX = np.expand_dims(valX / 255.0, axis=1)
testX = np.expand_dims(testX / 255.0, axis=1)

assert 0 <= np.max(trainX) <= 1

print('Train: X=%s, y=%s' % (trainX.shape, trainY.shape))
print('Val: X=%s, y=%s' % (valX.shape, valY.shape))
print('Test: X=%s, y=%s' % (testX.shape, testY.shape))

In [ ]:
class Transform(DataLoaderCallback):

    def __call__(self, data_loader: 'DataLoader', X_batch: np.ndarray, y_batch: np.ndarray):
        x = engine.tensor(X_batch, dtype=engine.float32, device=device)
        y = engine.tensor(y_batch, dtype=engine.long, device=device)
        return x, y

transform_cb = Transform()

train_loader = DataLoader(trainX, trainY, batch_size, transform=transform_cb)
val_loader = DataLoader(valX, valY, batch_size=256, transform=transform_cb)
test_loader = DataLoader(testX, testY, batch_size=256, transform=transform_cb)

print("Train batches:", len(train_loader), "| Val batches:", len(val_loader), "| Test batches:", len(test_loader))

In [ ]:
plt.figure(figsize=(7, 7))
for i in range(9):
    plt.subplot(330 + 1 + i)
    plt.imshow(trainX[i].squeeze(), cmap=plt.get_cmap('gray'))
plt.show()

In [ ]:
class CNN(nn.Module):

    def __init__(self):
        super().__init__()
        # 'same' padding keeps 28x28; each max pool halves it: 28 -> 14 -> 7
        conv1 = nn.Sequential(
            nn.Conv2d(1, 16, (3, 3), (1, 1), padding='same', bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.MaxPool2d(kernel_size=(2, 2))
        )
        conv2 = nn.Sequential(
            nn.Conv2d(16, 32, (3, 3), (1, 1), padding='same', bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.MaxPool2d(kernel_size=(2, 2))
        )
        self.cnn = nn.Sequential(conv1, conv2, nn.Flatten(), nn.Linear(32 * 7 * 7, 10))
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            with engine.no_grad():
                module.weight *= 0.1  # make the logits layer less confident
            nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Conv2d):
            nn.init.kaiming_normal_(module.weight, nonlinearity='relu')

    def forward(self, x: engine.Tensor):
        return self.cnn(x)

model = CNN()
model.to(device)
print("CNN Model:", model)
print("CNN Trainable Parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

trainer = Trainer(model, engine)
evaluator = Evaluator(accuracy=True, mode=Evaluator.MULTI_CLASS)
trainer.compile(loss_fn, optimizer, evaluator)
history = trainer.fit(train_loader, epochs=epochs, validation_loader=val_loader)

In [ ]:
trainer.plot(['accuracy', 'loss'], ylim=[0, 1])

In [ ]:
y_pred, y_true = trainer.test(test_loader)
evaluator.report(y_pred, y_true)

In [ ]:
plt.figure(figsize=(12, 12))
for i in range(9):
    plt.subplot(330 + 1 + i)
    plt.imshow(testX[i].squeeze(), cmap=plt.get_cmap('gray'))
    pred = y_pred[i].argmax(); label = y_true[i]
    plt.title(f"Pred: {pred} | Class: {label}")
plt.show()